# Business Entity Resolution — Amazon ML Challenge 2026

Runs the full pipeline end to end on a **SageMaker notebook instance**.

Each stage is a separate cell and writes checkpoints to disk, so you can re-run any
single cell without redoing the ones before it. Stage 1 is split per country for the
same reason — a failure costs one country, not five hours.

---

## Before you start: instance and volume

**Volume size is the one that bites.** SageMaker notebook instances default to a
**5 GB** EBS volume. This job needs the 2.3 GB dataset plus ~20 GB of intermediates.
Give the instance **at least 50 GB** when you create it (it cannot be changed later
without restarting the instance).

Recommended instance types:

| Instance | vCPU | RAM | Notes |
|---|---|---|---|
| `ml.m5.4xlarge` | 16 | 64 GB | Good balance. Comfortable headroom, ~2-3x faster than a 16GB laptop. |
| `ml.m5.2xlarge` | 8 | 32 GB | Cheaper, still fine. Expect roughly the times in `RUNBOOK.md`. |
| `ml.r5.2xlarge` | 8 | 64 GB | Memory-optimised; pick this if you raise `KEEP` or retrieval depth. |

`ml.t2/t3.*` will not work — they are burstable and will throttle to a crawl partway
through stage 1.

> **Cost note:** these are billed per hour for as long as the instance is *running*,
> not just while a cell executes. The full pipeline is several hours. Stop the instance
> when you are done.

The pipeline does not need a GPU. The only model is LightGBM on CPU.

## 1. Environment check

Run this first. It fails loudly if the instance is too small, which is much
cheaper than discovering it three hours into stage 1.

In [ ]:
import os, shutil, multiprocessing, platform, sys

cores = os.cpu_count()
try:
    ram_gb = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3
except (ValueError, AttributeError):
    ram_gb = float("nan")

print(f"platform : {platform.platform()}")
print(f"python   : {sys.version.split()[0]}")
print(f"cores    : {cores}")
print(f"RAM      : {ram_gb:.1f} GB")

problems = []
for label, path in [("home volume", os.path.expanduser("~")), ("/tmp", "/tmp")]:
    try:
        free_gb = shutil.disk_usage(path).free / 1024**3
        total_gb = shutil.disk_usage(path).total / 1024**3
        print(f"disk {label:<12}: {free_gb:.1f} GB free of {total_gb:.1f} GB  ({path})")
    except OSError as exc:
        print(f"disk {label:<12}: unreadable ({exc})")

if cores and cores < 4:
    problems.append(f"only {cores} cores - stage 1 will be very slow")
if ram_gb == ram_gb and ram_gb < 15:
    problems.append(f"only {ram_gb:.1f} GB RAM - the largest index alone needs ~7.5 GB")

home_free = shutil.disk_usage(os.path.expanduser("~")).free / 1024**3
if home_free < 30:
    problems.append(
        f"only {home_free:.1f} GB free on the home volume - need ~25 GB for dataset + "
        "intermediates. Recreate the notebook instance with a larger EBS volume.")

print()
if problems:
    print("PROBLEMS FOUND:")
    for p in problems:
        print("  ! " + p)
else:
    print("Environment looks fine.")

## 2. Dependencies

Versions are pinned to the ones the reported results were produced with.
Restart the kernel if pip upgrades `numpy` out from under an already-imported copy.

In [ ]:
import sys
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # Colab already ships numpy, scipy, scikit-learn, pandas and pyarrow. Forcing the
    # pinned versions from requirements.txt triggers a large reinstall and a mandatory
    # kernel restart for no benefit, so install only what is genuinely missing.
    %pip install -q "anyascii>=0.3.2" "rapidfuzz>=3.13" "lightgbm>=4.0"
else:
    %pip install -q -r student_resource/code/business_entity_resolution/requirements.txt
print("done")

In [ ]:
# Verify the imports the pipeline actually needs, and the one API most likely to be
# missing on an older rapidfuzz.
import numpy, scipy, sklearn, pyarrow, rapidfuzz, lightgbm, anyascii
from rapidfuzz import process

for m in (numpy, scipy, sklearn, pyarrow, rapidfuzz, lightgbm):
    print(f"  {m.__name__:<14} {getattr(m, '__version__', '?')}")

assert hasattr(process, "cpdist"), (
    "rapidfuzz.process.cpdist is missing - the feature stages depend on it. "
    "Upgrade with:  pip install -U rapidfuzz")
print("\nrapidfuzz.process.cpdist present - all imports OK")

## 3. Paths and the run helper

`REPO` must point at the repository root — the directory holding `student_resource/`.
Adjust it if you cloned somewhere else.

`WORK_DIR` holds ~20 GB of intermediates. Keep it on the EBS volume
(`/home/ec2-user/SageMaker/...`), **not** `/tmp` — on a notebook instance `/tmp` lives
on the small root volume and will fill up.

In [ ]:
import os, subprocess, sys, time, json
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # On Colab the dataset usually lives in Drive, but intermediates must NOT: work/
    # does ~20GB of random I/O and Drive's FUSE mount makes that painfully slow.
    # /content is local disk - fast, and lost when the session ends (which is fine,
    # every stage checkpoints and stage 1 is split per country).
    REPO     = Path("/content/Amazon_ML_Challeneg")
    DATA_DIR = Path("/content/drive/MyDrive/Amazon_ML_Challeneg/dataset")  # <-- adjust
    WORK_DIR = Path("/content/work")
    OUT_DIR  = REPO / "student_resource/output"
    if not DATA_DIR.exists():
        DATA_DIR = REPO / "student_resource/dataset"
else:
    # SageMaker notebook instance; falls back to wherever this notebook lives.
    REPO = Path("/home/ec2-user/SageMaker/Amazon_ML_Challeneg")
    if not REPO.exists():
        REPO = Path.cwd()
    DATA_DIR = REPO / "student_resource/dataset"
    OUT_DIR  = REPO / "student_resource/output"
    WORK_DIR = REPO / "work"

SRC = REPO / "student_resource/code/business_entity_resolution/src"
assert SRC.is_dir(), f"pipeline source not found at {SRC} - check REPO"
OUT_DIR.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)

ENV = {**os.environ,
       "BER_DATA_DIR": str(DATA_DIR),
       "BER_OUT_DIR":  str(OUT_DIR),
       "BER_WORK_DIR": str(WORK_DIR),
       "PYTHONUNBUFFERED": "1"}

def run(*args, cwd=SRC):
    """Run a pipeline stage, streaming its output live so long stages stay visible."""
    cmd = [sys.executable, *map(str, args)]
    print("$ " + " ".join(cmd) + "\n")
    t0 = time.time()
    p = subprocess.Popen(cmd, cwd=str(cwd), env=ENV, text=True, bufsize=1,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    mins = (time.time() - t0) / 60
    print(f"\n--- exit {p.returncode}, elapsed {mins:.1f} min ---")
    if p.returncode != 0:
        raise RuntimeError(f"stage failed with exit code {p.returncode}")

def sync_checkpoints(dest="/content/drive/MyDrive/Amazon_ML_Challeneg/work"):
    """Copy stage-0/1 checkpoints to Drive so a Colab disconnect does not lose them.

    Call after each stage-1 country. Restoring is just the same copy in reverse.
    """
    for sub in ("s0", "s1", "s3"):
        src = WORK_DIR / sub
        if src.is_dir():
            subprocess.run(["mkdir", "-p", f"{dest}/{sub}"], check=False)
            subprocess.run(f"cp -ru {src}/* {dest}/{sub}/", shell=True, check=False)
    print(f"checkpoints synced to {dest}")

print(f"environment : {'Colab' if IN_COLAB else 'SageMaker/local'}")
print(f"REPO        : {REPO}")
print(f"DATA_DIR    : {DATA_DIR}   exists={DATA_DIR.exists()}")
print(f"WORK_DIR    : {WORK_DIR}")
print(f"cores       : {os.cpu_count()}   (stage 1 picks its own worker count)")

## 4. Stage the dataset onto local disk (Colab only)

The dataset is ~2.3 GB and stage 0 reads all of it. Reading that through Drive's FUSE
mount is slow, so this extracts the archive to `/content` (local disk) once and points
`DATA_DIR` there.

Skip this cell on SageMaker, or if your TSVs are already on local disk.

In [ ]:
# Expects dataset.tar.gz in Drive (see COLAB_SETUP.md for how to build and upload it).
ARCHIVE    = Path("/content/drive/MyDrive/Amazon_ML_Challeneg/dataset.tar.gz")
LOCAL_DATA = Path("/content/dataset")

if IN_COLAB:
    from google.colab import drive
    if not Path("/content/drive").exists():
        drive.mount("/content/drive")

    if (LOCAL_DATA / "train").exists() and (LOCAL_DATA / "test").exists():
        print(f"already staged at {LOCAL_DATA}")
    elif ARCHIVE.exists():
        LOCAL_DATA.mkdir(parents=True, exist_ok=True)
        print(f"extracting {ARCHIVE} ({ARCHIVE.stat().st_size/1024**3:.2f} GB) ...")
        t0 = time.time()
        # --strip-components=1 drops the leading "dataset/" folder from the archive.
        subprocess.run(["tar", "xzf", str(ARCHIVE), "-C", str(LOCAL_DATA),
                        "--strip-components=1"], check=True)
        print(f"extracted in {(time.time()-t0)/60:.1f} min")
    else:
        raise SystemExit(f"archive not found at {ARCHIVE} - adjust the path above")

    DATA_DIR = LOCAL_DATA
    ENV["BER_DATA_DIR"] = str(DATA_DIR)
    print(f"\nDATA_DIR now {DATA_DIR}")
    for p in sorted(DATA_DIR.rglob("*.tsv")):
        print(f"  {p.relative_to(DATA_DIR)}  {p.stat().st_size/1024**2:.0f} MB")
else:
    print("not Colab - skipping")

## 5. Verify the dataset

The dataset is not in git. Upload it, or pull it from S3, so that the six TSVs
land under `student_resource/dataset/{train,test}/`.

To copy from S3 instead, run:
`!aws s3 sync s3://your-bucket/prefix/ {DATA_DIR}/`

In [ ]:
EXPECTED = {
    "train/train_source1.tsv": 2206822, "train/train_source2.tsv": 5034617,
    "train/train_source3.tsv": 5285604, "train/train_ground_truth.tsv": 2206822,
    "test/test_source1.tsv": 1732545, "test/test_source2.tsv": 4887274,
    "test/test_source3.tsv": 5082317,
}

missing = []
for rel, expected_lines in EXPECTED.items():
    path = DATA_DIR / rel
    if not path.exists():
        missing.append(rel)
        print(f"  MISSING  {rel}")
    else:
        size_mb = path.stat().st_size / 1024**2
        print(f"  ok       {rel:<32} {size_mb:8.1f} MB")

if missing:
    raise SystemExit(f"{len(missing)} dataset file(s) missing under {DATA_DIR}")
print("\nAll seven dataset files present.")

## 6. Stage 0 — normalize and shard by country

One streaming pass over all 2.3 GB. Folds accents, romanizes Indic scripts,
canonicalizes legal suffixes and street types, then writes per-country parquet shards.

Every later stage reads these shards, so the expensive normalization is paid once.

**~5 min.** Skip this cell if `work/s0/` is already populated.

In [ ]:
run("s0_prepare.py")

## 7. Stage 1 — candidate generation (the long pole)

Three character n-gram TF-IDF indexes per country (name, address, name+address),
unioned and reranked down to 60 candidates per Source-1 entity.

Each country is a separate call and writes its own `.npz`, so a failure or a restart
only costs that one country. **Several hours in total.** Watch the `[rss N.NNGB]`
figures: if they approach your instance's RAM, lower `WORKERS`.

In [ ]:
# Smallest first, so any problem shows up in 20 minutes rather than 3 hours.
run("s1_block.py", "train", "--countries", "India",  "--workers", WORKERS)

In [ ]:
run("s1_block.py", "train", "--countries", "US", "--workers", WORKERS)

In [ ]:
run("s1_block.py", "test", "--countries", "France", "--workers", WORKERS)

In [ ]:
run("s1_block.py", "test", "--countries", "US", "--workers", WORKERS)

In [ ]:
# Largest shard in the job: 4.72M references, ~7.5 GB resident for the index alone.
# On a 32 GB instance this is fine; on anything smaller drop --workers to 2.
run("s1_block.py", "test", "--countries", "India", "--workers", WORKERS)

## 8. Blocking quality — check this before going further

Recall here is a hard ceiling: a true match that blocking never proposes can never be
predicted, no matter how good the model is.

Reference figures measured on a 5,000-entity India sample against the full 4.13M
reference pool: **pair recall 91.65%**, entity recall 79.40%.

If the full-scale number comes out far below ~90%, stop and investigate — it is not
worth spending hours on stages 2-4 over a broken candidate set.

In [ ]:
run("eval_blocking.py")

## 9. Stage 2 — pairwise features

~30 float32 similarity features per candidate pair, written straight into a memmap so
the ~12.5 GB test matrix is never resident. Internally this uses all available cores.

**~15 min.**

In [ ]:
run("s2_features.py", "train")
run("s2_features.py", "test")

## 10. Stage 3 — train the model and tune the decision rule

LightGBM over the pairwise features, then a joint sweep of `(threshold, margin)` scored
with the *real* macro-F_0.5 on an entity-disjoint holdout — singletons included, exactly
as the leaderboard counts them.

**The `BEST: macro F0.5 = ...` line is your validation score.** It is the number to put
in `Documentation_template.md` and the best predictor you have of the leaderboard.

`--neg-frac` controls how many negatives are kept for the fit. The default 0.25 is
a 16GB compromise; **on a host with 32GB or more, pass `--neg-frac 1.0`** to train on
every negative, which gives a better-calibrated probability and therefore a better
threshold. The full training matrix is only ~2.2GB. Drop to `0.15` if RAM is tight.

In [ ]:
# On a host with >=32GB RAM, train on every negative: better calibration, ~2.2GB matrix.
import os
NEG_FRAC = "1.0" if (os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 1024**3) > 30 else "0.25"
print(f"using --neg-frac {NEG_FRAC}\n")
run("s3_train.py", "--neg-frac", NEG_FRAC)

In [ ]:
with open(WORK_DIR / "s3" / "decision.json") as f:
    decision = json.load(f)
print(json.dumps({k: v for k, v in decision.items() if k != "features"}, indent=2))

## 11. Stage 4 — write the submission files

Scores every test candidate, applies the tuned rule, and writes both TSVs. Streams
`test_source1.tsv` to guarantee exactly one row per Source-1 entity, in file order,
including France and including entities with no candidates.

**~10 min.**

In [ ]:
run("s4_predict.py")

## 12. Validate — do this before every upload

`--check-ids` is the only check that catches a corrupted index-to-entity_id mapping,
which would otherwise silently cost you a whole submission. It costs ~16 s.

Expect `PASS` and exactly **1,732,544** rows in each file.

In [ ]:
run("utils/validate_submission.py",
    "--matching",  OUT_DIR / "matching_results.tsv",
    "--candidate", OUT_DIR / "candidate_pairs.tsv",
    "--test-dir",  DATA_DIR / "test",
    "--check-ids",
    cwd=REPO / "student_resource")

## 13. Package the final submission zip

Builds the archive in the structure the challenge requires:

```
<team>_submission.zip
├── output/{matching_results,candidate_pairs}.tsv
├── code/business_entity_resolution/{src,README.md,requirements.txt}
└── Documentation_template.md
```

Fill in the `[FILL]` placeholders in `Documentation_template.md` with the F_0.5 and
decision rule from stage 3 **before** running this.

In [ ]:
import zipfile

TEAM = "your_team_name"          # <-- set this
zip_path = REPO / f"{TEAM}_submission.zip"
student = REPO / "student_resource"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for name in ("matching_results.tsv", "candidate_pairs.tsv"):
        z.write(OUT_DIR / name, f"output/{name}")
    base = student / "code/business_entity_resolution"
    for f in sorted(base.rglob("*")):
        if f.is_file() and "__pycache__" not in f.parts:
            z.write(f, f"code/business_entity_resolution/{f.relative_to(base)}")
    z.write(student / "Documentation_template.md", "Documentation_template.md")

print(f"wrote {zip_path}  ({zip_path.stat().st_size / 1024**2:.1f} MB)\n")
with zipfile.ZipFile(zip_path) as z:
    for n in z.namelist():
        print("  " + n)